# Beyond the Basics — SDK Patterns for Real-World Implementations

*Optional notebook. Assumes you have completed notebooks 01 and 02.*

Topics:
1. Multi-turn conversations — `to_input_list()` and streaming REPL
2. State management — `context=` and `RunContextWrapper`
3. Lifecycle hooks — minimal `RunHooks`
4. Deterministic vs stochastic orchestration (SDK version)
5. SDK reference — `handoff_description`, `tool_use_behavior`, `final_output_as()`, `ModelSettings`

In [ ]:
import os
import asyncio
import numpy as np
import faiss
import logging
from openai import OpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import (
    Agent, Runner, RunConfig, function_tool, handoff,
    RunHooks, RunContextWrapper, RawResponsesStreamEvent,
)
from pydantic import BaseModel

# Suppress verbose HTTP logs from the OpenAI client
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Setup complete")

In [ ]:
COURSE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def _embed(texts):
    r = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in r.data], dtype=np.float32)

_vecs = _embed(COURSE_TEXTS)
_index = faiss.IndexFlatL2(_vecs.shape[1])
_index.add(_vecs)

@function_tool
def search_course_materials(query: str) -> str:
    """Search ANLP course materials for information relevant to the query. Returns 2 passages."""
    q = _embed([query]).reshape(1, -1)
    _, idx = _index.search(q, 2)
    return "\n\n".join(f"[{i+1}] {COURSE_TEXTS[j]}" for i, j in enumerate(idx[0]))

chat_agent = Agent(
    name="Course Assistant",
    instructions="You are a helpful NLP course assistant. Be concise — 2-3 sentences.",
    tools=[search_course_materials],
    model="gpt-4o-mini",
)
print(f"Index ready: {_index.ntotal} vectors")

## 1. Multi-turn Conversations

In notebooks 01 and 02 every `Runner.run_sync()` call was independent.
For a real conversation the agent needs to remember what was said before.

`result.to_input_list()` returns the full message history — user messages, agent responses,
and tool call records — formatted as the input list for the next run.

In [ ]:
# Turn 1 — standalone run
result1 = Runner.run_sync(chat_agent, "What is a vector embedding?")
print(f"Turn 1:\n{result1.final_output}\n")

# to_input_list() serialises the full exchange: user message + agent response + any tool records
history = result1.to_input_list()
print(f"History after turn 1: {len(history)} messages")

# Turn 2 — agent sees the prior exchange
history.append({"role": "user", "content": "How is it used in RAG?"})
result2 = Runner.run_sync(chat_agent, history)
print(f"\nTurn 2:\n{result2.final_output}")

print(f"\nHistory after turn 2: {len(result2.to_input_list())} messages")

### Streaming REPL

Combine `to_input_list()` with `Runner.run_streamed()` to build a multi-turn terminal
conversation where tokens appear in real time.

`Runner.run_streamed()` is async — we wrap the loop in an `async def` and `await` it.
Jupyter supports top-level `await`, so no `asyncio.run()` is needed.

In [ ]:
async def streaming_repl():
    history = []
    print("Course Assistant (streaming) — type 'quit' to exit\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit", ""):
            print("Goodbye!")
            break

        history.append({"role": "user", "content": user_input})
        print("Assistant: ", end="", flush=True)

        streamed = Runner.run_streamed(chat_agent, history)
        async for event in streamed.stream_events():
            if (isinstance(event, RawResponsesStreamEvent)
                    and isinstance(event.data, ResponseTextDeltaEvent)):
                print(event.data.delta, end="", flush=True)
        print()  # newline after response

        # Preserve full history — agent remembers prior turns
        history = streamed.to_input_list()

await streaming_repl()

## 2. State Management with `context=`

`to_input_list()` preserves conversation history. But sometimes you need to track
*application* state across turns — a turn counter, accumulated results, a flag —
without using global variables.

Pass any Python object to `Runner.run_sync(context=...)`.
It is available inside hooks via `context.context`.

In [ ]:
class SessionContext:
    """Tracks run-level state across agent turns."""
    def __init__(self):
        self.turns: int = 0
        self.tool_calls_made: int = 0

# context= makes this object available to hooks via RunContextWrapper.context
# It does NOT get passed to the LLM — it is Python-layer state only
ctx = SessionContext()
print(f"Before run: turns={ctx.turns}, tool_calls={ctx.tool_calls_made}")

result = Runner.run_sync(
    chat_agent,
    "What is cosine similarity?",
    context=ctx,
)
print(f"After run:  turns={ctx.turns}, tool_calls={ctx.tool_calls_made}")
print(f"(Context unchanged until hooks update it — see next section)")

## 3. Lifecycle Hooks — `RunHooks`

`RunHooks` lets you attach callbacks to agent lifecycle events without touching business logic.
Subclass it and override only the methods you need.

Events available:
| Event | Fires when |
|---|---|
| `on_agent_start` | An agent begins executing |
| `on_agent_end` | An agent produces its final output |
| `on_handoff` | Control transfers between agents |
| `on_tool_start` | A tool is about to be called |
| `on_tool_end` | A tool call has returned |

Pass your hooks instance via `hooks=` on `Runner.run_sync()`.

In [ ]:
from typing import Any

class LoggingHooks(RunHooks):
    """Minimal hooks: log transitions and update SessionContext."""

    def on_agent_start(self, context: RunContextWrapper, agent) -> None:
        print(f"  [hook] → {agent.name} started")
        if context.context:
            context.context.turns += 1

    def on_agent_end(self, context: RunContextWrapper, agent, output: Any) -> None:
        print(f"  [hook] ← {agent.name} finished (turn {context.context.turns if context.context else '?'})")

    def on_tool_start(self, context: RunContextWrapper, agent, tool) -> None:
        print(f"  [hook] ⚙ tool called: {tool.name}")
        if context.context:
            context.context.tool_calls_made += 1

ctx = SessionContext()

print("Running with hooks:\n")
result = Runner.run_sync(
    chat_agent,
    "What is cosine similarity and how does FAISS use it?",
    context=ctx,
    hooks=LoggingHooks(),
)
print(f"\nAnswer: {result.final_output[:120]}...")
print(f"\nContext tracked: turns={ctx.turns}, tool_calls={ctx.tool_calls_made}")

### Circuit breaker via hooks

You can raise `MaxTurnsExceeded` inside `on_agent_end` to stop the run early:

```python
from agents import MaxTurnsExceeded

class BudgetHooks(RunHooks):
    def __init__(self, max_turns: int):
        self.max_turns = max_turns

    def on_agent_end(self, context, agent, output):
        if context.context and context.context.turns >= self.max_turns:
            raise MaxTurnsExceeded(f"Budget of {self.max_turns} turns reached")
```

This is the correct pattern for enforcing hard limits on long-running agentic loops.

## 4. Deterministic vs Stochastic Orchestration — SDK Version

In notebook 01 we showed the evaluator-optimizer as a raw API loop.
Here we show the same pattern using the Agents SDK — and contrast the two control strategies.

**Deterministic:** programmer writes the loop, calls generator then evaluator in sequence.
**Stochastic:** an orchestrator agent decides when to generate, evaluate, and stop.

Same outcome. Different locus of control. This is the same distinction as agentic RAG in notebooks 01 and 02.

In [ ]:
class EvalResult(BaseModel):
    score: int    # 1–10
    feedback: str
    approved: bool  # True if score >= 8

generator_agent = Agent(
    name="Generator",
    instructions=(
        "Generate a clear one-paragraph explanation of the given topic "
        "suitable for NLP Master's students."
    ),
    model="gpt-4o-mini",
)

evaluator_agent = Agent(
    name="Evaluator",
    instructions=(
        "Evaluate this explanation: score 1–10, one-sentence feedback, "
        "and set approved=True only if score >= 8."
    ),
    output_type=EvalResult,
    model="gpt-4o-mini",
)

In [ ]:
# Deterministic: programmer controls every step
topic = "What is cosine similarity and why does it matter for semantic search?"
history = [{"role": "user", "content": f"Explain: {topic}"}]

print("=== Deterministic loop ===")
for round_num in range(1, 4):
    gen = Runner.run_sync(generator_agent, history)
    draft = gen.final_output
    print(f"\nRound {round_num} draft: {draft[:100]}...")

    eval_r = Runner.run_sync(evaluator_agent, f"Evaluate:\n\n{draft}")
    fb: EvalResult = eval_r.final_output
    print(f"Score: {fb.score}/10 | {fb.feedback}")

    if fb.approved:
        print(f"✓ Approved at round {round_num}")
        break

    history = gen.to_input_list()
    history.append({"role": "user", "content": f"Improve based on: {fb.feedback}"})

In [ ]:
# Stochastic: orchestrator LLM decides when to generate, evaluate, and stop
reflection_orchestrator = Agent(
    name="ReflectionOrchestrator",
    instructions=(
        "Generate an explanation using 'generate'. "
        "Evaluate it using 'evaluate'. "
        "If score < 8, improve the explanation and re-evaluate. "
        "Stop only when approved=True."
    ),
    tools=[
        generator_agent.as_tool("generate", "Generate an explanation for the given topic"),
        evaluator_agent.as_tool("evaluate", "Score an explanation; returns approved=True if score >= 8"),
    ],
    model="gpt-4o-mini",
)

print("=== Stochastic orchestrator ===")
result = Runner.run_sync(reflection_orchestrator, f"Explain: {topic}")
print(result.final_output)
print(f"\nSteps taken by orchestrator: {len(result.new_items)}")

## 5. SDK Reference — Patterns You Will See in Real Code

Brief working examples of SDK features that appear frequently in production code
but were not covered in notebooks 01 and 02.

In [ ]:
from agents import ModelSettings

# ── handoff_description ──────────────────────────────────────────────────────
# Describes this agent to the router — exactly like a tool docstring, but for handoffs.
# The router LLM reads this to decide which agent to hand off to.
billing_agent = Agent(
    name="Billing Specialist",
    handoff_description="Handles billing, invoice, and payment questions.",  # ← router reads this
    instructions="You are a billing specialist. Be precise and offer concrete next steps.",
    model="gpt-4o-mini",
)

# ── ModelSettings ─────────────────────────────────────────────────────────────
# Per-agent temperature, max_tokens — override the model default for this agent only.
precise_agent = Agent(
    name="Precise Answerer",
    instructions="Give exact numerical answers only. No prose.",
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.0, max_tokens=50),
)

r = Runner.run_sync(precise_agent, "What is 0.85 to the power of 10, rounded to 4 decimal places?")
print(f"Precise agent: {r.final_output}")

# ── final_output_as() ─────────────────────────────────────────────────────────
# Cast final_output to a specific Pydantic type — useful when output_type is set.
class Summary(BaseModel):
    key_points: list[str]

summary_agent = Agent(
    name="Summariser",
    instructions="Summarise in exactly 3 bullet points.",
    output_type=Summary,
    model="gpt-4o-mini",
)
r = Runner.run_sync(summary_agent, "What is RAG?")
typed: Summary = r.final_output_as(Summary)
print(f"\nSummary type: {type(typed).__name__}")
for point in typed.key_points:
    print(f"  • {point}")

# ── tool_use_behavior ─────────────────────────────────────────────────────────
# Controls what happens after a tool call returns.
# "run_llm_again" (default): agent continues reasoning after every tool result.
# "stop_on_first_tool": agent stops after the first tool call and returns the raw tool output.
print("""
tool_use_behavior options:
  "run_llm_again"      — agent keeps reasoning after tool results (default)
  "stop_on_first_tool" — agent stops after first tool call; tool output IS the final output
  list of tool names   — stop only when these specific tools are called
""")